# Generalized full-frame paper-theater notebook

This version keeps the original pipeline structure, but removes the most fragile scene-specific pieces.

## What changed
- automatic scene naming from the input image
- optional Gemini / OpenAI vision scene analysis
- dynamic Florence query generation instead of a fixed object list
- generalized label normalization into semantic families
- generalized fallback-mask restriction by semantic family instead of exact labels
- generalized layer prompts based on layer role and front/behind context
- multiple debugging cells to inspect:
  - scene analysis JSON
  - generated Florence queries
  - detection overlays
  - segmentation masks
  - depth map
  - layer summaries
  - focus masks sent to the image model

## Recommended use
1. Put one input image in the `data/input` folder.
2. Set `IMAGE_PATH` in the config cell.
3. Run the notebook top to bottom.
4. Inspect the debug cells before generating the final full-frame layers.

## Patch notes

This patched version keeps the AI scene-analysis step, but stabilizes the generalized pipeline by:
- constraining Florence queries to broad canonical object families
- normalizing labels before downstream grouping
- dropping explicit ground/detail detections such as path, grass, and bag
- reordering final layer contexts with depth-aware postprocessing
- keeping fallback pixels broad and preventing them from flooding subject layers
- returning to an anchor-first show mask closer to the stable scene2 behavior

In [ ]:
%cd /content
!git clone https://github.com/UrbinaDan/PaperTheater_ai_Pipeline.git
%cd PaperTheater_ai_Pipeline

!pip install -q numpy scipy matplotlib opencv-python-headless pillow shapely svgwrite cairosvg tqdm networkx pyyaml requests google-genai openai accelerate bitsandbytes einops sentencepiece transformers==4.49.0

import os
from getpass import getpass

%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
!wget -q -P checkpoints https://raw.githubusercontent.com/facebookresearch/sam2/main/sam2_configs/sam2_hiera_s.yaml

%cd /content
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd Depth-Anything-V2
!mkdir -p checkpoints
!wget -q -P checkpoints https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth

%cd /content/PaperTheater_ai_Pipeline

In [ ]:
import os
from pathlib import Path
from getpass import getpass

# =========================
# USER CONFIG
# =========================

IMAGE_PATH = "/content/PaperTheater_ai_Pipeline/data/input/scene_2.jpg"
VISION_PROVIDER = "gemini"           # "gemini", "openai", or "none"
IMAGE_GENERATION_PROVIDER = "gemini" # final generation currently uses Gemini
USE_VISION_LLM = True
USE_LLM_LAYER_GUIDANCE = False

GEMINI_TEXT_MODEL = "gemini-2.5-flash"
GEMINI_IMAGE_MODEL = "gemini-3.1-flash-image-preview"
OPENAI_TEXT_MODEL = "gpt-5"

MAX_QUERIES = 8
DEBUG_SHOW_EACH_OBJECT_MASK = True
DEBUG_SHOW_EACH_LAYER_MASK = True

STYLE_NORMALIZATION_PROMPT = """
Render this as a stylized Japanese paper theater / kamishibai layer.

GLOBAL STYLE RULES:
- flat paper-cut illustration
- simplified silhouettes and clean edges
- muted earthy palette
- minimal texture
- no photorealism
- no realistic skin or fur rendering
- no detailed lighting or harsh shadows
- consistent abstraction across all layers
- all layers should look like they wifere made by the same artist
- preserve the original composition and scale inside the full canvas
- output should feel like a clean cuttable layer for paper theater production
""".strip()

if USE_VISION_LLM and VISION_PROVIDER == "gemini" and "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

if USE_VISION_LLM and VISION_PROVIDER == "openai" and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if IMAGE_GENERATION_PROVIDER == "gemini" and "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key for image generation: ")
USE_RAW_DEPTH_FALLBACK_FOR_GEMINI = True
DEBUG_SHOW_LEFTOVER_ASSIGNMENT = True

In [ ]:
# =========================
# IMPORTS + DEPTH-BAND CONFIG
# =========================

import os
import io
import re
import ast
import json
import time
import base64
import shutil
import importlib
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from google import genai
from google.genai.errors import ServerError, ClientError
from openai import OpenAI

from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image, save_mask
from src.florence_parser import FlorenceParser
from src.sam2_segmenter import SAM2Segmenter
from src.depth_anything_runner import DepthRunner
from src.occlusion_heuristic import heuristic_complete
from src.mask_cleanup import cleanup_mask

import src.scene_builder
importlib.reload(src.scene_builder)
from src.scene_builder import parse_florence_boxes

paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"]) if os.environ.get("GEMINI_API_KEY") else None
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"]) if os.environ.get("OPENAI_API_KEY") else None

# =========================
# NEW DEPTH-BAND PIPELINE CONFIG
# =========================
# Keep the first three setup/config cells unchanged.
# These options override/extend the old semantic-layer pipeline below.

MAX_DETECTION_QUERIES = 50          # Florence can try many objects now.
NUM_DEPTH_LAYERS = 5                # Desired paper-theater depth bands.
MIN_OBJECT_AREA_RATIO = 0.0015      # Remove tiny noisy detections.
DETECTION_IOU_DEDUPE = 0.82
BAND_MIN_AREA_RATIO = 0.002         # Drop empty/tiny depth bands after grouping.
USE_DEPTH_BAND_LAYERS = True        # Main new behavior.
HARD_ENFORCE_BLACK_AFTER_GENERATION = True

# Derived output paths
image_path = IMAGE_PATH
SCENE_NAME = Path(image_path).stem

BASE_INTERMEDIATE_DIR = Path("/content/PaperTheater_ai_Pipeline/data/intermediate")
SCENE_DIR = BASE_INTERMEDIATE_DIR / SCENE_NAME
FULLFRAME_OUTPUT_DIR = SCENE_DIR / "depth_band_fullframe_outputs"
DEBUG_DIR = SCENE_DIR / "debug_depth_band"
MASK_DIR = SCENE_DIR / "depth_band_masks"

SCENE_ANALYSIS_PATH = SCENE_DIR / f"{SCENE_NAME}_scene_analysis.json"
QUERY_DEBUG_PATH = SCENE_DIR / f"{SCENE_NAME}_depth_band_queries.json"
LAYER_GUIDANCE_PATH = SCENE_DIR / f"{SCENE_NAME}_depth_band_layer_guidance.json"

for p in [SCENE_DIR, FULLFRAME_OUTPUT_DIR, DEBUG_DIR, MASK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("SCENE_NAME:", SCENE_NAME)
print("image_path:", image_path)
print("SCENE_DIR:", SCENE_DIR)
print("FULLFRAME_OUTPUT_DIR:", FULLFRAME_OUTPUT_DIR)


# =========================
# BASIC HELPERS
# =========================

def show_image(img, title="", figsize=(8, 12), cmap=None):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

def image_to_pil(image_rgb):
    if isinstance(image_rgb, Image.Image):
        return image_rgb.convert("RGB")
    if isinstance(image_rgb, np.ndarray):
        if image_rgb.ndim == 2:
            image_rgb = np.stack([image_rgb] * 3, axis=-1)
        if image_rgb.shape[-1] == 4:
            image_rgb = image_rgb[..., :3]
        return Image.fromarray(image_rgb.astype(np.uint8)).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(image_rgb)}")

def normalize_to_rgb_array(img_obj):
    if isinstance(img_obj, np.ndarray):
        arr = img_obj
        if arr.ndim == 2:
            return np.stack([arr] * 3, axis=-1).astype(np.uint8)
        if arr.shape[-1] == 4:
            return arr[..., :3].astype(np.uint8)
        return arr.astype(np.uint8)
    if isinstance(img_obj, Image.Image):
        return np.array(img_obj.convert("RGB"))
    if isinstance(img_obj, (bytes, bytearray)):
        return np.array(Image.open(io.BytesIO(img_obj)).convert("RGB"))
    if hasattr(img_obj, "image_bytes") and img_obj.image_bytes is not None:
        raw = img_obj.image_bytes
        if isinstance(raw, str):
            raw = base64.b64decode(raw)
        return np.array(Image.open(io.BytesIO(raw)).convert("RGB"))
    if hasattr(img_obj, "numpy_view"):
        arr = np.array(img_obj.numpy_view())
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)
        elif arr.shape[-1] == 4:
            arr = arr[..., :3]
        return arr.astype(np.uint8)
    raise TypeError(f"Unsupported image object type: {type(img_obj)}")

def save_bool_mask(mask, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)

def mask_to_rgba_fullframe(image_rgb, mask):
    alpha = (mask.astype(np.uint8)) * 255
    return np.dstack([image_rgb, alpha])

def safe_json_loads(text, default=None):
    if default is None:
        default = {}
    if text is None:
        return default
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except Exception:
        pass
    for pattern in [r"\{.*\}", r"\[.*\]"]:
        m = re.search(pattern, text, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                try:
                    return ast.literal_eval(m.group(0))
                except Exception:
                    pass
    return default

def clean_query_text(q):
    q = str(q).strip().lower()
    q = re.sub(r"^[\-*\d\.\)\s]+", "", q)
    q = q.replace("_", " ")
    q = re.sub(r"\s+", " ", q).strip()
    return q[:80]

def extract_caption_text(caption_obj):
    if isinstance(caption_obj, dict):
        for key in ["<MORE_DETAILED_CAPTION>", "caption", "text", "scene_summary"]:
            val = caption_obj.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
        vals = [str(v).strip() for v in caption_obj.values() if isinstance(v, str) and str(v).strip()]
        return " ".join(vals).strip()
    if isinstance(caption_obj, str):
        return caption_obj.strip()
    if caption_obj is None:
        return ""
    return str(caption_obj).strip()

def dedupe_keep_order(items):
    out = []
    seen = set()
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def clamp_box(bbox, image_shape):
    h, w = image_shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1 = max(0, min(w - 1, x1))
    x2 = max(0, min(w - 1, x2))
    y1 = max(0, min(h - 1, y1))
    y2 = max(0, min(h - 1, y2))
    if x2 <= x1:
        x2 = min(w - 1, x1 + 1)
    if y2 <= y1:
        y2 = min(h - 1, y1 + 1)
    return [x1, y1, x2, y2]

def box_iou(boxA, boxB):
    ax1, ay1, ax2, ay2 = boxA
    bx1, by1, bx2, by2 = boxB
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    areaA = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    areaB = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

def box_contains(boxA, boxB, margin=8):
    ax1, ay1, ax2, ay2 = boxA
    bx1, by1, bx2, by2 = boxB
    return (ax1 - margin <= bx1 <= bx2 <= ax2 + margin) and (ay1 - margin <= by1 <= by2 <= ay2 + margin)

def area_fraction(mask):
    return float(mask.sum()) / float(mask.size) if mask.size else 0.0

def mask_bbox(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return [0, 0, 1, 1]
    return [int(xs.min()), int(ys.min()), int(xs.max() + 1), int(ys.max() + 1)]

def dominant_labels(labels, max_items=8):
    counts = Counter(labels)
    return [k for k, _ in counts.most_common(max_items)]

def dynamic_kernel_from_mask(mask, min_k=9, max_k=151, frac=0.08):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return min_k
    bw = xs.max() - xs.min() + 1
    bh = ys.max() - ys.min() + 1
    ref = max(bw, bh)
    k = int(max(min_k, min(max_k, ref * frac)))
    if k % 2 == 0:
        k += 1
    return k

def expand_mask(mask, kernel_size=None):
    if kernel_size is None:
        kernel_size = dynamic_kernel_from_mask(mask.astype(np.uint8))
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    return cv2.dilate(mask.astype(np.uint8), kernel, iterations=1) > 0

In [ ]:
# =========================
# LABEL NORMALIZATION + QUERY GENERATION
# =========================

LABEL_CANONICAL_MAP = {
    # human-like / figure
    "people": "figure", "family": "figure", "child": "figure", "children": "figure",
    "man": "figure", "woman": "figure", "boy": "figure", "girl": "figure",
    "human": "figure", "tourist": "figure", "person": "figure",
    "figure": "figure", "figures": "figure", "human figure": "figure",
    "statue": "figure", "sculpture": "figure",

    # animals
    "deer": "animal", "dog": "animal", "cat": "animal", "horse": "animal",
    "bird": "animal", "cow": "animal", "sheep": "animal",

    # vegetation
    "tree": "trees", "trees": "trees", "forest": "trees",
    "branch": "trees", "branches": "trees", "foliage": "trees", "plant": "trees",
    "flowering tree": "trees", "cherry blossom tree": "trees",
    "blossom tree": "trees", "cherry blossom": "trees",

    # smaller vegetation details
    "flower": "flowers", "flowers": "flowers", "tulip": "flowers",
    "tulips": "flowers", "flower bed": "flowers", "tulip garden": "flowers",
    "hedge": "hedge", "hedges": "hedge", "bush": "hedge",
    "bushes": "hedge", "shrub": "hedge", "shrubs": "hedge",

    # architecture
    "building": "building", "brick building": "building", "temple": "building",
    "pagoda": "building", "church": "building", "cathedral": "building",
    "house": "building", "tower": "building", "turret": "building",
    "spire": "building", "shrine": "building", "gate": "building",
    "bridge": "building", "roof": "building", "window": "building", "windows": "building",

    # background / landscape
    "cloud": "sky", "clouds": "sky", "sky": "sky",
    "mountain": "mountain", "mountains": "mountain", "hill": "mountain", "hills": "mountain",
    "water": "water", "river": "water", "lake": "water", "sea": "water", "ocean": "water",

    # ground / low-priority detail
    "grass": "ground", "path": "ground", "road": "ground",
    "field": "ground", "lawn": "ground", "floor": "ground",
    "bag": "detail",
}

DROP_LABELS = {"detail"}  # Ground can be useful for depth-band grouping, so do not drop it here.

FAMILY_BY_LABEL = {
    "sky": "sky_background",
    "mountain": "distant_landscape",
    "water": "water",
    "building": "architecture",
    "trees": "vegetation",
    "flowers": "vegetation_detail",
    "hedge": "vegetation_detail",
    "figure": "figure_subject",
    "animal": "animal_subject",
    "ground": "ground_plane",
}

def canonicalize_label(label):
    l = clean_query_text(label)
    return LABEL_CANONICAL_MAP.get(l, l)

def normalized_family_from_label(label):
    return FAMILY_BY_LABEL.get(canonicalize_label(label), "object")

def call_vision_text_llm(image_rgb, prompt, provider="gemini"):
    pil_img = image_to_pil(image_rgb)

    if provider == "gemini":
        if gemini_client is None:
            raise RuntimeError("Gemini client is not available.")
        response = gemini_client.models.generate_content(
            model=GEMINI_TEXT_MODEL,
            contents=[prompt, pil_img],
        )
        return getattr(response, "text", "") or ""

    if provider == "openai":
        if openai_client is None:
            raise RuntimeError("OpenAI client is not available.")
        buffered = io.BytesIO()
        pil_img.save(buffered, format="PNG")
        b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
        response = openai_client.responses.create(
            model=OPENAI_TEXT_MODEL,
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": f"data:image/png;base64,{b64}"},
                ],
            }],
        )
        return getattr(response, "output_text", "") or ""

    raise ValueError(f"Unsupported provider: {provider}")

def analyze_scene_with_llm(image_rgb, fallback_caption="", provider="gemini"):
    prompt = f"""
Analyze this image for a generalized paper-theater depth-band extraction pipeline.

Return STRICT JSON only with this exact schema:
{{
  "scene_summary": "1-3 sentence summary",
  "style_tags": ["tag1", "tag2"],
  "important_objects": ["object1", "object2"],
  "framing_elements": ["element1", "element2"],
  "background_elements": ["element1", "element2"],
  "foreground_elements": ["element1", "element2"],
  "candidate_detection_queries": ["query1", "query2"],
  "likely_indoor_or_outdoor": "indoor|outdoor|mixed",
  "notes_for_layering": ["note1", "note2"]
}}

Rules:
- Candidate queries should include as many visible object groups as useful for depth grouping.
- Include repeated object types if they appear at different depths, such as foreground trees and background trees.
- Prefer short nouns Florence can detect: sky, clouds, building, tower, tree, flowers, hedge, person, figure, animal, ground, path.
- Do not add explanations outside JSON.

Fallback caption:
{fallback_caption}
""".strip()

    raw = call_vision_text_llm(image_rgb, prompt, provider=provider)
    parsed = safe_json_loads(raw, default={})
    if not isinstance(parsed, dict):
        parsed = {}
    parsed.setdefault("scene_summary", fallback_caption or "")
    parsed.setdefault("style_tags", [])
    parsed.setdefault("important_objects", [])
    parsed.setdefault("framing_elements", [])
    parsed.setdefault("background_elements", [])
    parsed.setdefault("foreground_elements", [])
    parsed.setdefault("candidate_detection_queries", [])
    parsed.setdefault("likely_indoor_or_outdoor", "mixed")
    parsed.setdefault("notes_for_layering", [])
    return parsed

BASE_QUERY_BANK = [
    # background
    "sky", "clouds", "mountain", "hills", "water",
    # architecture
    "building", "brick building", "temple", "pagoda", "church", "cathedral",
    "tower", "turret", "spire", "house", "roof", "gate", "bridge",
    # vegetation
    "tree", "trees", "forest", "branches", "foliage", "flowering tree",
    "cherry blossom tree", "blossom tree", "flowers", "flower bed", "tulips",
    "hedge", "bushes", "shrubs", "grass", "path", "ground",
    # subjects
    "person", "people", "family", "child", "children", "human figure",
    "man", "woman", "statue", "sculpture", "animal", "deer", "dog", "cat", "bird",
]

def gather_text_blob(scene_analysis, fallback_caption=""):
    terms = []
    if isinstance(scene_analysis, dict):
        for key in [
            "scene_summary",
            "candidate_detection_queries",
            "important_objects",
            "background_elements",
            "foreground_elements",
            "framing_elements",
            "notes_for_layering",
        ]:
            vals = scene_analysis.get(key, [])
            if isinstance(vals, str):
                vals = [vals]
            terms.extend(vals)
    terms.append(extract_caption_text(fallback_caption))
    return " | ".join(clean_query_text(x) for x in terms if x).lower()

def generate_many_queries(scene_analysis, fallback_caption="", max_queries=50):
    text_blob = gather_text_blob(scene_analysis, fallback_caption)
    queries = []

    # 1) Use LLM-proposed objects first.
    for key in ["candidate_detection_queries", "important_objects", "background_elements", "foreground_elements", "framing_elements"]:
        vals = scene_analysis.get(key, []) if isinstance(scene_analysis, dict) else []
        if isinstance(vals, str):
            vals = [vals]
        for v in vals:
            q = clean_query_text(v)
            if q:
                queries.append(q)

    # 2) Add canonical queries if related words appear.
    for q in BASE_QUERY_BANK:
        q_clean = clean_query_text(q)
        q_words = q_clean.split()
        if q_clean in text_blob or any(w in text_blob for w in q_words):
            queries.append(q_clean)

    # 3) Always try broadly useful scene categories.
    queries.extend([
        "sky", "building", "tree", "flowers", "person", "figure",
        "animal", "ground", "path", "foreground", "background"
    ])

    # 4) Clean and cap.
    queries = [clean_query_text(q) for q in queries]
    queries = [q for q in queries if q and q not in {"background", "foreground"}]
    queries = dedupe_keep_order(queries)
    return queries[:max_queries]

def dedupe_boxes_depth_pipeline(boxes, image_shape, iou_threshold=0.82):
    clean = []
    H, W = image_shape[:2]
    img_area = float(H * W)

    for b in boxes:
        label = canonicalize_label(b.get("label", "object"))
        if label in DROP_LABELS:
            continue

        bbox = clamp_box(b["bbox"], image_shape)
        x1, y1, x2, y2 = bbox
        area = max(1, (x2 - x1) * (y2 - y1))
        area_ratio = area / img_area

        if area_ratio < MIN_OBJECT_AREA_RATIO:
            continue

        # Sky should be broad; tiny cloud boxes canonicalized as sky are not a full sky layer.
        if label == "sky" and area_ratio < 0.04:
            label = "clouds"

        keep = True
        replace_idx = None

        for i, prev in enumerate(clean):
            prev_label = prev["label"]
            prev_bbox = prev["bbox"]
            iou = box_iou(prev_bbox, bbox)

            # Only dedupe strongly overlapping boxes of same canonical label.
            if prev_label == label and iou >= iou_threshold:
                prev_area = max(1, (prev_bbox[2]-prev_bbox[0])*(prev_bbox[3]-prev_bbox[1]))
                if area > prev_area:
                    replace_idx = i
                keep = False
                break

            # Containment cleanup for same label only.
            if prev_label == label and (box_contains(prev_bbox, bbox) or box_contains(bbox, prev_bbox)):
                prev_area = max(1, (prev_bbox[2]-prev_bbox[0])*(prev_bbox[3]-prev_bbox[1]))
                # Keep both if the smaller is not tiny, because same object class can appear at multiple depths.
                smaller = min(area, prev_area)
                larger = max(area, prev_area)
                if smaller / larger < 0.20:
                    if area > prev_area:
                        replace_idx = i
                    keep = False
                    break

        if replace_idx is not None:
            clean[replace_idx] = {"label": label, "bbox": bbox}
        elif keep:
            clean.append({"label": label, "bbox": bbox})

    return clean

def labels_present_from_boxes(boxes):
    return {canonicalize_label(b.get("label", "object")) for b in boxes}

In [ ]:
# =========================
# LOAD IMAGE + MODELS
# =========================

image = load_image(image_path, max_side=cfg.image_max_side)

show_image(image, title=f"{SCENE_NAME} — input", figsize=(8, 12))
print("image shape:", image.shape)

florence = FlorenceParser(cfg.florence_model)
segmenter = SAM2Segmenter(cfg.sam2_config, cfg.sam2_checkpoint)
depth_runner = DepthRunner(cfg.depth_encoder)


# =========================
# SCENE ANALYSIS
# =========================

caption = florence.get_dense_caption(image)
caption_text = extract_caption_text(caption)

print("Florence caption:")
print(caption)

scene_analysis = {
    "scene_summary": caption_text,
    "style_tags": [],
    "important_objects": [],
    "framing_elements": [],
    "background_elements": [],
    "foreground_elements": [],
    "candidate_detection_queries": [],
    "likely_indoor_or_outdoor": "mixed",
    "notes_for_layering": [],
}

if USE_VISION_LLM and VISION_PROVIDER in {"gemini", "openai"}:
    try:
        scene_analysis = analyze_scene_with_llm(
            image_rgb=image,
            fallback_caption=caption_text,
            provider=VISION_PROVIDER,
        )
    except Exception as e:
        print("Vision LLM analysis failed. Falling back to Florence caption only.")
        print("Error:", repr(e))

if not isinstance(scene_analysis.get("scene_summary", ""), str):
    scene_analysis["scene_summary"] = caption_text

with open(SCENE_ANALYSIS_PATH, "w", encoding="utf-8") as f:
    json.dump(scene_analysis, f, indent=2, ensure_ascii=False)

print("caption_text:")
print(caption_text)
print("Scene analysis:")
print(json.dumps(scene_analysis, indent=2, ensure_ascii=False))


# =========================
# MANY FLORENCE DETECTION QUERIES
# =========================

queries = generate_many_queries(
    scene_analysis,
    fallback_caption=caption_text,
    max_queries=MAX_DETECTION_QUERIES,
)

print("Generated queries:", len(queries))
for i, q in enumerate(queries, 1):
    print(f"{i:02d}. {q}")

all_boxes = []
raw_detection_dump = {}

def run_detection_queries(query_list, stage_name="base"):
    stage_boxes = []
    for q in query_list:
        try:
            det_q = florence.get_open_vocab_detection(image, q)
            boxes_q = parse_florence_boxes(det_q, image.shape)
        except Exception as e:
            det_q = {"error": repr(e)}
            boxes_q = []

        raw_detection_dump[f"{stage_name}:{q}"] = {
            "raw": det_q,
            "parsed": boxes_q,
        }
        print(f"\n[{stage_name.upper()}] QUERY:", q)
        print("PARSED:", boxes_q)
        stage_boxes.extend(boxes_q)
    return stage_boxes

all_boxes.extend(run_detection_queries(queries, stage_name="many"))
boxes = dedupe_boxes_depth_pipeline(all_boxes, image.shape, iou_threshold=DETECTION_IOU_DEDUPE)

# If obvious families from the text are missing, add a small recovery pass.
text_blob = gather_text_blob(scene_analysis, caption_text)
present = labels_present_from_boxes(boxes)
recovery_queries = []

recovery_by_label = {
    "sky": ["sky", "clouds"],
    "building": ["building", "brick building", "temple", "church", "tower", "roof"],
    "trees": ["tree", "trees", "flowering tree", "cherry blossom tree", "foliage"],
    "flowers": ["flowers", "tulips", "flower bed"],
    "hedge": ["hedge", "bushes", "shrubs"],
    "figure": ["person", "people", "family", "child", "human figure", "statue", "sculpture"],
    "animal": ["animal", "deer", "dog", "cat", "bird"],
    "ground": ["ground", "grass", "path", "road", "field"],
    "mountain": ["mountain", "hills"],
}

wanted_patterns = {
    "sky": r"\b(sky|cloud|clouds)\b",
    "building": r"\b(building|brick building|temple|pagoda|church|cathedral|tower|turret|spire|roof|house|shrine|gate)\b",
    "trees": r"\b(tree|trees|forest|branch|branches|foliage|flowering tree|cherry blossom|blossom)\b",
    "flowers": r"\b(flower|flowers|tulip|tulips|flower bed|garden)\b",
    "hedge": r"\b(hedge|hedges|bush|bushes|shrub|shrubs)\b",
    "figure": r"\b(person|people|family|child|children|man|woman|figure|statue|sculpture)\b",
    "animal": r"\b(animal|deer|dog|cat|horse|bird)\b",
    "ground": r"\b(ground|grass|path|road|field|lawn|floor)\b",
    "mountain": r"\b(mountain|mountains|hill|hills)\b",
}

for label, pattern in wanted_patterns.items():
    if re.search(pattern, text_blob) and label not in present:
        recovery_queries.extend(recovery_by_label[label])

recovery_queries = [q for q in dedupe_keep_order(recovery_queries) if q not in queries]

if recovery_queries:
    print("\nRecovery queries:", len(recovery_queries))
    for i, q in enumerate(recovery_queries, 1):
        print(f"{i:02d}. {q}")
    all_boxes.extend(run_detection_queries(recovery_queries, stage_name="recovery"))
    boxes = dedupe_boxes_depth_pipeline(all_boxes, image.shape, iou_threshold=0.78)

if not boxes:
    emergency_queries = ["sky", "building", "tree", "person", "animal", "ground"]
    print("Using emergency fallback queries:", emergency_queries)
    all_boxes.extend(run_detection_queries(emergency_queries, stage_name="emergency"))
    boxes = dedupe_boxes_depth_pipeline(all_boxes, image.shape, iou_threshold=0.78)

with open(QUERY_DEBUG_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "queries": queries,
        "recovery_queries": recovery_queries,
        "boxes": boxes,
    }, f, indent=2, ensure_ascii=False)

print("\nFINAL DEDUPED BOXES:")
print(boxes)
print("num boxes:", len(boxes))

fig, ax = plt.subplots(figsize=(8, 12))
ax.imshow(image)
for b in boxes:
    x1, y1, x2, y2 = b["bbox"]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, max(0, y1 - 6), b["label"], color="yellow", fontsize=9, backgroundcolor="black")
ax.axis("off")
ax.set_title("Florence detections — many deduped")
plt.show()

with open(DEBUG_DIR / "raw_detection_dump.json", "w", encoding="utf-8") as f:
    json.dump(raw_detection_dump, f, indent=2, ensure_ascii=False, default=str)

In [ ]:
# =========================
# SEGMENTATION + DEPTH
# =========================

def normalize_segmented_objects(segmented, depth_map, image_shape):
    """
    Robust wrapper around SAM output.

    Keeps individual detections instead of merging all objects with the same label.
    This is important because the same semantic class can appear at different depths.
    """
    H, W = image_shape[:2]
    objects = []

    for idx, obj in enumerate(segmented):
        if not isinstance(obj, dict):
            continue

        label = canonicalize_label(obj.get("label", obj.get("query", "object")))
        if label in DROP_LABELS:
            continue

        mask = obj.get("mask", None)
        if mask is None:
            # Some segmenters may use a different key.
            for alt_key in ["segmentation", "binary_mask"]:
                if alt_key in obj:
                    mask = obj[alt_key]
                    break

        if mask is None:
            continue

        mask = np.array(mask) > 0
        if mask.shape != (H, W):
            mask = np.array(
                Image.fromarray(mask.astype(np.uint8) * 255).resize(
                    (W, H),
                    Image.Resampling.NEAREST,
                )
            ) > 0

        area = int(mask.sum())
        if area < int(H * W * MIN_OBJECT_AREA_RATIO):
            continue

        bbox = obj.get("bbox", mask_bbox(mask))
        bbox = clamp_box(bbox, image_shape)
        vals = depth_map[mask]
        rep_depth = float(np.median(vals)) if vals.size else float(np.median(depth_map))

        objects.append({
            "id": len(objects),
            "label": label,
            "raw_label": obj.get("label", label),
            "family": normalized_family_from_label(label),
            "bbox": bbox,
            "mask": mask,
            "area": area,
            "rep_depth": rep_depth,
        })

    return objects

def lightly_cleanup_object_masks(objects):
    """
    Gentle cleanup only. Does not merge by semantic label.
    """
    cleaned = []
    for obj in objects:
        mask = obj["mask"] > 0

        # Fill tiny holes / remove small specks while keeping object separation.
        mask_u8 = mask.astype(np.uint8)
        kernel = np.ones((3, 3), np.uint8)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel, iterations=1)
        mask = mask_u8 > 0

        new_obj = dict(obj)
        new_obj["mask"] = mask
        new_obj["bbox"] = mask_bbox(mask)
        new_obj["area"] = int(mask.sum())
        cleaned.append(new_obj)

    return cleaned

segmented = segmenter.segment_boxes(image, boxes)
depth = depth_runner.infer(image)

stable_objects = normalize_segmented_objects(segmented, depth, image.shape)
stable_objects = lightly_cleanup_object_masks(stable_objects)

# Recompute IDs after cleanup.
for i, obj in enumerate(stable_objects):
    obj["id"] = i
    obj["rep_depth"] = float(np.median(depth[obj["mask"] > 0])) if (obj["mask"] > 0).any() else float(np.median(depth))

print("num segmented objects:", len(segmented))
print("num stable depth objects:", len(stable_objects))

show_image(depth, title="Depth map", figsize=(8, 12), cmap="viridis")

fig, ax = plt.subplots(figsize=(8, 12))
ax.imshow(image)
for obj in stable_objects:
    x1, y1, x2, y2 = obj["bbox"]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.text(
        x1,
        max(0, y1 - 6),
        f"{obj['id']}:{obj['label']} [{obj['family']}] d={obj['rep_depth']:.3f}",
        color="white",
        fontsize=8,
        backgroundcolor="black",
    )
ax.axis("off")
ax.set_title("Stable depth objects")
plt.show()

object_rows = []
for obj in stable_objects:
    object_rows.append({
        "id": obj["id"],
        "label": obj["label"],
        "family": obj["family"],
        "area": obj["area"],
        "area_frac": round(obj["area"] / (image.shape[0] * image.shape[1]), 5),
        "rep_depth": round(obj["rep_depth"], 6),
        "bbox": obj["bbox"],
    })

object_df = pd.DataFrame(object_rows).sort_values("rep_depth")
display(object_df)

if DEBUG_SHOW_EACH_OBJECT_MASK:
    for obj in stable_objects:
        mask = obj["mask"] > 0
        overlay = image.copy()
        overlay[mask] = (0.65 * overlay[mask] + 0.35 * np.array([255, 80, 80])).astype(np.uint8)
        show_image(
            overlay,
            title=f"Object {obj['id']} — {obj['label']} — {obj['family']} — depth {obj['rep_depth']:.3f}",
            figsize=(6, 9),
        )


# =========================
# DEPTH-BAND LAYER CONTEXTS
# =========================

def infer_depth_direction_from_objects(objects, depth_map):
    """
    Returns convention and a normalized nearness function.
    """
    far_vals = []
    near_vals = []

    for obj in objects:
        fam = obj.get("family", "object")
        d = float(obj.get("rep_depth", 0.0))
        if fam in {"sky_background", "distant_landscape"}:
            far_vals.append(d)
        if fam in {"figure_subject", "animal_subject", "vegetation_detail", "ground_plane"}:
            near_vals.append(d)

    if far_vals and near_vals:
        far_med = float(np.median(far_vals))
        near_med = float(np.median(near_vals))
        return "larger_is_closer" if near_med > far_med else "smaller_is_closer"

    # Fallback: Depth Anything often appears larger/warmer for closer in your debug maps.
    return "larger_is_closer"

def compute_nearness_values(objects, depth_direction):
    depths = np.array([float(obj["rep_depth"]) for obj in objects], dtype=np.float32)
    d_min = float(depths.min()) if len(depths) else 0.0
    d_max = float(depths.max()) if len(depths) else 1.0
    d_range = max(1e-6, d_max - d_min)

    for obj in objects:
        d = float(obj["rep_depth"])
        if depth_direction == "larger_is_closer":
            obj["near_score"] = (d - d_min) / d_range
        else:
            obj["near_score"] = (d_max - d) / d_range

def estimate_sky_allowed_zone_from_objects(objects, image_shape):
    H, W = image_shape[:2]
    sky_masks = [obj["mask"] > 0 for obj in objects if obj.get("family") == "sky_background"]
    if not sky_masks:
        yy = np.arange(H)[:, None]
        return np.repeat(yy <= int(H * 0.40), W, axis=1)

    sky_union = np.logical_or.reduce(sky_masks)
    ys, xs = np.where(sky_union)
    if len(ys) == 0:
        yy = np.arange(H)[:, None]
        return np.repeat(yy <= int(H * 0.40), W, axis=1)

    sky_bottom = int(np.percentile(ys, 90))
    margin = int(H * 0.025)
    yy = np.arange(H)[:, None]
    return np.repeat(yy <= min(H - 1, sky_bottom + margin), W, axis=1)

def choose_depth_band_count(objects, requested=5):
    if len(objects) <= 3:
        return max(2, len(objects))
    return int(max(3, min(requested, len(objects))))

def build_depth_band_layer_contexts(objects, depth_map, image_shape, num_layers=5):
    """
    Core new layer builder:
    - Keep many objects.
    - Compute object depth.
    - Split object depth range into N far->near bands.
    - Merge objects inside each depth band into one paper-theater layer.
    """
    H, W = image_shape[:2]
    if not objects:
        raise ValueError("No stable objects available for depth-band grouping.")

    depth_direction = infer_depth_direction_from_objects(objects, depth_map)
    compute_nearness_values(objects, depth_direction)

    n_bands = choose_depth_band_count(objects, num_layers)

    # Force sky to the farthest band, then group everything else by nearness.
    for obj in objects:
        if obj.get("family") == "sky_background":
            obj["band_index"] = 0
        else:
            band = int(np.floor(obj["near_score"] * n_bands))
            band = max(0, min(n_bands - 1, band))
            obj["band_index"] = band

    # Ensure at least one architecture/landscape band does not get swallowed by sky.
    # This is generic: only sky family is forced into band 0.
    band_to_objects = defaultdict(list)
    for obj in objects:
        band_to_objects[obj["band_index"]].append(obj)

    contexts = []
    for band_idx in range(n_bands):
        band_objects = band_to_objects.get(band_idx, [])
        if not band_objects:
            continue

        ownership_mask = np.zeros((H, W), dtype=bool)
        visible_mask = np.zeros((H, W), dtype=bool)

        labels = []
        families = []
        object_ids = []
        depths = []
        near_scores = []

        for obj in band_objects:
            m = obj["mask"] > 0
            ownership_mask |= m
            visible_mask |= m
            labels.append(obj["label"])
            families.append(obj["family"])
            object_ids.append(obj["id"])
            depths.append(float(obj["rep_depth"]))
            near_scores.append(float(obj["near_score"]))

        if area_fraction(ownership_mask) < BAND_MIN_AREA_RATIO:
            continue

        dominant_family = Counter(families).most_common(1)[0][0]
        label_summary = dominant_labels(labels, max_items=8)

        # Make names stable and interpretable.
        depth_name = ["background", "far_midground", "midground", "near_midground", "foreground", "frontmost"]
        suffix = depth_name[min(band_idx, len(depth_name) - 1)]
        content_name = "_".join(label_summary[:3]) if label_summary else "objects"
        layer_name = f"depth_{band_idx:02d}_{suffix}_{content_name}"

        contexts.append({
            "order": len(contexts),
            "layer_name": layer_name,
            "family": "depth_band",
            "band_index": band_idx,
            "dominant_family": dominant_family,
            "labels": label_summary,
            "all_labels": labels,
            "families": sorted(set(families)),
            "object_ids": object_ids,
            "ownership_mask": ownership_mask,
            "visible_mask": visible_mask,
            "rep_depth": float(np.median(depths)) if depths else float(np.median(depth_map)),
            "near_score": float(np.median(near_scores)) if near_scores else 0.0,
            "scene_caption": scene_analysis.get("scene_summary", caption_text),
            "depth_direction": depth_direction,
        })

    # Sort far -> near by band index.
    contexts = sorted(contexts, key=lambda c: c["band_index"])

    for i, ctx in enumerate(contexts):
        ctx["order"] = i
        front_layers = contexts[i + 1:]
        ctx["front_layer_names"] = [x["layer_name"] for x in front_layers]
        front_labels = []
        for x in front_layers:
            front_labels.extend(x.get("labels", []))
        ctx["front_layer_labels"] = dedupe_keep_order(front_labels)

    print("Depth direction inferred:", depth_direction)
    print("Depth-band layers far -> near:")
    for ctx in contexts:
        print({
            "order": ctx["order"],
            "layer_name": ctx["layer_name"],
            "band_index": ctx["band_index"],
            "dominant_family": ctx["dominant_family"],
            "labels": ctx["labels"],
            "object_ids": ctx["object_ids"],
            "rep_depth": round(ctx["rep_depth"], 6),
            "near_score": round(ctx["near_score"], 4),
            "area": int(ctx["ownership_mask"].sum()),
        })

    return contexts

layer_contexts = build_depth_band_layer_contexts(
    stable_objects,
    depth,
    image.shape,
    num_layers=NUM_DEPTH_LAYERS,
)

layer_rows = []
for ctx in layer_contexts:
    layer_rows.append({
        "order": ctx["order"],
        "layer_name": ctx["layer_name"],
        "band_index": ctx["band_index"],
        "dominant_family": ctx["dominant_family"],
        "labels": ", ".join(ctx["labels"]),
        "families": ", ".join(ctx["families"]),
        "object_ids": ", ".join(map(str, ctx["object_ids"])),
        "area_frac": round(area_fraction(ctx["ownership_mask"]), 4),
        "rep_depth": round(ctx["rep_depth"], 6),
        "near_score": round(ctx["near_score"], 4),
    })

layer_df = pd.DataFrame(layer_rows)
display(layer_df)

In [ ]:
# =========================
# DEPTH FALLBACK + CONTROL MASKS
# =========================

def build_unassigned_pixel_mask(layer_contexts, image_shape):
    H, W = image_shape[:2]
    assigned = np.zeros((H, W), dtype=bool)
    for ctx in layer_contexts:
        assigned |= (ctx["ownership_mask"] > 0)
    return ~assigned

def assign_leftover_pixels_to_nearest_depth_band(layer_contexts, depth_map):
    H, W = depth_map.shape
    unassigned = build_unassigned_pixel_mask(layer_contexts, (H, W))

    fallback_masks = {
        ctx["layer_name"]: np.zeros((H, W), dtype=bool)
        for ctx in layer_contexts
    }
    assignment_index = np.full((H, W), -1, dtype=np.int32)

    if len(layer_contexts) == 0:
        return fallback_masks, unassigned, assignment_index

    layer_depths = np.array([float(ctx["rep_depth"]) for ctx in layer_contexts], dtype=np.float32)
    layer_names = [ctx["layer_name"] for ctx in layer_contexts]

    # Prevent sky/background band from stealing lower image pixels.
    sky_allowed_zone = estimate_sky_allowed_zone_from_objects(stable_objects, (H, W))
    sky_layer_indices = [
        i for i, ctx in enumerate(layer_contexts)
        if "sky" in ctx.get("labels", []) or ctx.get("dominant_family") == "sky_background"
    ]

    ys, xs = np.where(unassigned)
    if len(ys) == 0:
        return fallback_masks, unassigned, assignment_index

    pixel_depths = depth_map[ys, xs].astype(np.float32)
    dist = np.abs(pixel_depths[:, None] - layer_depths[None, :])
    sorted_idx = np.argsort(dist, axis=1)

    for k in range(len(ys)):
        y = ys[k]
        x = xs[k]
        chosen_idx = None

        for candidate_idx in sorted_idx[k]:
            candidate_idx = int(candidate_idx)
            if candidate_idx in sky_layer_indices and not sky_allowed_zone[y, x]:
                continue
            chosen_idx = candidate_idx
            break

        if chosen_idx is None:
            chosen_idx = int(sorted_idx[k][0])

        fallback_masks[layer_names[chosen_idx]][y, x] = True
        assignment_index[y, x] = chosen_idx

    return fallback_masks, unassigned, assignment_index

def get_layer_support_mask(layer_context, depth_fallback_masks):
    visible_mask = layer_context["visible_mask"] > 0
    fallback_mask = depth_fallback_masks[layer_context["layer_name"]] > 0
    return visible_mask | fallback_mask

def build_control_masks_for_layer(layer_context, ordered_contexts, depth_fallback_masks):
    order = layer_context["order"]
    layer_name = layer_context["layer_name"]

    visible_mask = layer_context["visible_mask"] > 0
    fallback_mask = depth_fallback_masks[layer_name] > 0
    current_support = visible_mask | fallback_mask

    # In depth-band mode, each layer is a slice, so show all support.
    show_mask = current_support.copy()

    H, W = visible_mask.shape
    front_visible = np.zeros((H, W), dtype=bool)
    behind_support = np.zeros((H, W), dtype=bool)

    for ctx in ordered_contexts:
        if ctx["layer_name"] == layer_name:
            continue

        if ctx["order"] > order:
            front_visible |= (ctx["visible_mask"] > 0)
        elif ctx["order"] < order:
            behind_support |= get_layer_support_mask(ctx, depth_fallback_masks)

    # The farthest band should have no black background behind it.
    if order == 0:
        black_mask = np.zeros((H, W), dtype=bool)
    else:
        black_mask = behind_support.copy()

    black_mask = black_mask & (~show_mask)
    beige_mask = ~(show_mask | black_mask)

    return {
        "anchor_mask": visible_mask,
        "fallback_mask": fallback_mask,
        "current_support": current_support,
        "front_visible": front_visible,
        "behind_support": behind_support,
        "show_mask": show_mask,
        "black_mask": black_mask,
        "beige_mask": beige_mask,
    }

def apply_focus_mask_fullframe(image_rgb, show_mask, black_mask, beige_value=(235, 235, 235)):
    if isinstance(beige_value, int):
        beige_value = (beige_value, beige_value, beige_value)
    h, w = show_mask.shape
    out = np.full((h, w, 3), beige_value, dtype=np.uint8)
    out[black_mask > 0] = 0
    out[show_mask > 0] = image_rgb[show_mask > 0]
    return out

def summarize_leftover_assignment(layer_contexts, depth_fallback_masks, unassigned_pixels_mask, assignment_index):
    print("\nLeftover pixel assignment summary:")
    total_leftover = int(unassigned_pixels_mask.sum())
    print("total leftover pixels:", total_leftover)
    if total_leftover == 0:
        return

    for idx_layer, ctx in enumerate(layer_contexts):
        layer_name = ctx["layer_name"]
        assigned_count = int((assignment_index == idx_layer).sum())
        fallback_count = int(depth_fallback_masks[layer_name].sum())
        print({
            "layer_name": layer_name,
            "assigned_count": assigned_count,
            "fallback_count": fallback_count,
        })

def show_leftover_assignment_map(layer_contexts, image_rgb, unassigned_pixels_mask, assignment_index):
    h, w = unassigned_pixels_mask.shape
    color_map = np.zeros((h, w, 3), dtype=np.uint8)
    rng = np.random.default_rng(0)
    palette = rng.integers(40, 255, size=(max(1, len(layer_contexts)), 3), dtype=np.uint8)

    for idx_layer, _ctx in enumerate(layer_contexts):
        color_map[assignment_index == idx_layer] = palette[idx_layer]

    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    axes[0].imshow(image_rgb)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    axes[1].imshow(unassigned_pixels_mask, cmap="gray")
    axes[1].set_title("Leftover pixels")
    axes[1].axis("off")

    axes[2].imshow(color_map)
    axes[2].set_title("Leftover -> nearest depth-band layer")
    axes[2].axis("off")
    plt.show()

def show_mask_debug(
    image_rgb,
    layer_name,
    anchor_mask,
    fallback_mask,
    current_support,
    front_visible,
    behind_support,
    show_mask,
    black_mask,
    focused_input,
    unassigned_mask=None,
):
    h, w = anchor_mask.shape
    comp = np.zeros((h, w, 3), dtype=np.uint8)
    comp[anchor_mask] = [80, 140, 255]
    comp[fallback_mask] = [220, 80, 220]
    comp[front_visible] = [80, 220, 220]
    comp[behind_support] = [220, 80, 80]

    if unassigned_mask is None:
        unassigned_mask = ~(show_mask | black_mask)

    fig, axes = plt.subplots(2, 4, figsize=(20, 18))
    axes = axes.ravel()

    panels = [
        (image_rgb, f"{layer_name} - original", None),
        (current_support, "current_support", "gray"),
        (front_visible, "front_visible", "gray"),
        (behind_support, "behind_support (black)", "gray"),
        (comp, "blue=anchor magenta=fallback cyan=front red=behind", None),
        (focused_input, "focused_input sent to image model", None),
        (unassigned_mask, "beige/editable mask", "gray"),
        (black_mask, "black forbidden mask", "gray"),
    ]

    for ax, (img, title, cmap) in zip(axes, panels):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

depth_fallback_masks, unassigned_pixels_mask, leftover_assignment_index = assign_leftover_pixels_to_nearest_depth_band(
    layer_contexts,
    depth,
)

print("Using depth-band leftover assignment.")
show_image(unassigned_pixels_mask, title="Unassigned pixels after object masks", figsize=(8, 12), cmap="gray")
summarize_leftover_assignment(
    layer_contexts,
    depth_fallback_masks,
    unassigned_pixels_mask,
    leftover_assignment_index,
)

if DEBUG_SHOW_LEFTOVER_ASSIGNMENT:
    show_leftover_assignment_map(
        layer_contexts,
        image,
        unassigned_pixels_mask,
        leftover_assignment_index,
    )

ordered_contexts = sorted(layer_contexts, key=lambda x: x["order"])

for ctx in ordered_contexts:
    layer_name = ctx["layer_name"]
    mask_pack = build_control_masks_for_layer(ctx, ordered_contexts, depth_fallback_masks)
    focused_input = apply_focus_mask_fullframe(
        image_rgb=image,
        show_mask=mask_pack["show_mask"],
        black_mask=mask_pack["black_mask"],
        beige_value=235,
    )

    print("=" * 70)
    print("LAYER:", layer_name)
    print("band_index:", ctx.get("band_index"))
    print("dominant_family:", ctx.get("dominant_family"))
    print("labels:", ctx.get("labels"))
    print("families:", ctx.get("families"))
    print("object_ids:", ctx.get("object_ids"))
    print("anchor sum:", int(mask_pack["anchor_mask"].sum()))
    print("fallback sum:", int(mask_pack["fallback_mask"].sum()))
    print("current_support sum:", int(mask_pack["current_support"].sum()))
    print("front_visible sum:", int(mask_pack["front_visible"].sum()))
    print("behind_support sum:", int(mask_pack["behind_support"].sum()))
    print("show_mask sum:", int(mask_pack["show_mask"].sum()))
    print("black_mask sum:", int(mask_pack["black_mask"].sum()))
    print("beige_mask sum:", int(mask_pack["beige_mask"].sum()))
    print("show ∩ black:", int((mask_pack["show_mask"] & mask_pack["black_mask"]).sum()))

    if DEBUG_SHOW_EACH_LAYER_MASK:
        show_mask_debug(
            image_rgb=image,
            layer_name=layer_name,
            anchor_mask=mask_pack["anchor_mask"],
            fallback_mask=mask_pack["fallback_mask"],
            current_support=mask_pack["current_support"],
            front_visible=mask_pack["front_visible"],
            behind_support=mask_pack["behind_support"],
            show_mask=mask_pack["show_mask"],
            black_mask=mask_pack["black_mask"],
            focused_input=focused_input,
            unassigned_mask=mask_pack["beige_mask"],
        )

In [ ]:
# =========================
# DEPTH-BAND PROMPTS + GENERATION
# =========================

def build_depth_band_guidance(layer_context, scene_analysis):
    labels = layer_context.get("labels", [])
    families = layer_context.get("families", [])
    dominant_family = layer_context.get("dominant_family", "object")
    order = layer_context.get("order", 0)
    total = len(layer_contexts)

    if order == 0 or "sky" in labels:
        return {
            "layer_goal": "reconstruct the farthest background depth slice",
            "density_guidance": "Keep distant/background regions broad and calm.",
            "style_guidance": "Use clean layered paper-cut shapes.",
        }

    if order == total - 1:
        return {
            "layer_goal": "reconstruct the closest foreground depth slice",
            "density_guidance": "Preserve foreground anchors and continue nearby support naturally.",
            "style_guidance": "Use crisp cuttable foreground silhouettes.",
        }

    return {
        "layer_goal": "reconstruct this depth-band slice of the same scene",
        "density_guidance": "Fill allowed areas only with content consistent with this depth band.",
        "style_guidance": "Use consistent paper-theater style.",
    }

def build_fullframe_prompt(layer_context, layer_guidance):
    labels = layer_context.get("labels", [])
    all_labels = layer_context.get("all_labels", [])
    families = layer_context.get("families", [])
    dominant_family = layer_context.get("dominant_family", "object")
    layer_name = layer_context["layer_name"]
    order = layer_context.get("order", 0)
    total_layers = len(layer_contexts)

    labels_text = ", ".join(labels) if labels else "mixed scene elements"
    families_text = ", ".join(families) if families else dominant_family
    front_labels = ", ".join(layer_context.get("front_layer_labels", []))

    if order == 0 or "sky" in labels:
        critical_text = """
CRITICAL BACKGROUND RULES:
- This is the farthest background depth slice.
- Reconstruct only the content supported by the visible anchors in this far background layer.
- If the visible anchors are sky/clouds, fill editable regions with smooth sky/cloud continuation.
- Do not reveal mask silhouettes.
- Do not add foreground trees, people, animals, flowers, or architecture unless they are already colored anchors in this layer.
""".strip()
    else:
        critical_text = """
CRITICAL DEPTH-BAND RULES:
- Reconstruct the SAME scene at this depth range.
- This is a depth slice, not a single object cutout.
- Colored pixels are anchors for all content that belongs to this depth band.
- Fill white/beige allowed regions by continuing the same materials and objects visible in the anchors.
- If anchors show building, continue building.
- If anchors show trees or foliage, continue trees or foliage.
- If anchors show grass, ground, flowers, path, or hedge, continue that same material.
- Do not redesign the scene.
- Do not invent new focal objects that were not supported by anchors.
- Do not duplicate people, animals, figures, or objects.
- Do not create a second scene or repeated stacked composition.
- Do not reveal mask boundaries.
""".strip()

    return f"""
Generate a FULL-FRAME paper-theater layer from the provided control image.

Scene context:
{layer_context.get("scene_caption", "")}

Target layer:
- layer name: {layer_name}
- layer type: depth-band slice
- depth order: {order + 1} of {total_layers} from far to near
- visible labels in this band: {labels_text}
- semantic families in this band: {families_text}
- front/closer layer labels: {front_labels}

{STYLE_NORMALIZATION_PROMPT}

INPUT INTERPRETATION:
- Colored pixels are real anchor pixels from the original image.
- Preserve colored anchor pixels as the strongest reference for position, scale, color family, and style.
- Black regions are forbidden/protected regions from deeper/farther layers. Do not paint into black regions.
- White, gray, or beige regions are allowed editable regions for this depth band.
- The mask colors are technical guides only.
- Do not reveal, trace, outline, or stylize the mask shape.
- Do not create checkerboard patterns, UI texture, transparency grids, or placeholder backgrounds.

{critical_text}

LAYER GOAL:
- {layer_guidance.get("layer_goal", "reconstruct this depth-band layer cleanly")}

ADDITIONAL GUIDANCE:
- {layer_guidance.get("density_guidance", "Continue visible environment naturally inside allowed areas.")}
- {layer_guidance.get("style_guidance", "Keep a consistent paper-theater style.")}

IMPORTANT RULES:
1. Preserve exact scene alignment with the original image.
2. Preserve the scale, silhouette, and position of colored anchor pixels.
3. Do not redesign the scene.
4. Do not invent unrelated focal objects.
5. Do not paint into black regions.
6. Fully reconstruct allowed white/beige regions according to the visible anchors in this same depth band.
7. Do not leave large blank white or beige holes unless the true layer is empty/sky there.
8. Keep shapes flat, clean, simple, and cuttable.
9. Do not reveal mask boundaries.
10. The output should look like one finished full-frame paper-theater depth layer.

OUTPUT GOAL:
A complete full-frame depth-band layer that preserves anchors, respects black forbidden regions, and naturally reconstructs the allowed areas for this depth range.
""".strip()

layer_guidance_map = {
    ctx["layer_name"]: build_depth_band_guidance(ctx, scene_analysis)
    for ctx in ordered_contexts
}

with open(LAYER_GUIDANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(layer_guidance_map, f, indent=2, ensure_ascii=False)

for ctx in ordered_contexts:
    prompt_preview = build_fullframe_prompt(
        layer_context=ctx,
        layer_guidance=layer_guidance_map[ctx["layer_name"]],
    )
    print("\n" + "=" * 80)
    print(ctx["layer_name"])
    print("=" * 80)
    print(prompt_preview[:2500])


def gemini_edit(image_rgb, prompt, model_name, max_retries=4):
    if gemini_client is None:
        raise RuntimeError("Gemini client is not available.")
    pil_img = image_to_pil(image_rgb)
    last_err = None

    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model_name,
                contents=[prompt, pil_img],
            )

            if hasattr(response, "generated_images") and response.generated_images:
                img_obj = response.generated_images[0].image
                return normalize_to_rgb_array(img_obj)

            if hasattr(response, "candidates") and response.candidates:
                for cand in response.candidates:
                    content = getattr(cand, "content", None)
                    if content is None:
                        continue
                    for part in getattr(content, "parts", []):
                        inline_data = getattr(part, "inline_data", None)
                        if inline_data is not None and getattr(inline_data, "data", None):
                            raw = inline_data.data
                            if isinstance(raw, str):
                                raw = base64.b64decode(raw)
                            return np.array(Image.open(io.BytesIO(raw)).convert("RGB"))
                        if hasattr(part, "as_image"):
                            maybe_img = part.as_image()
                            return normalize_to_rgb_array(maybe_img)

            if hasattr(response, "text"):
                print("No usable image found. Gemini text preview:", repr(getattr(response, "text", None))[:800])
            raise ValueError("Gemini response did not contain a usable image")

        except ServerError as e:
            last_err = e
            wait_s = min(60, 5 * (2 ** attempt))
            print(f"Gemini ServerError on attempt {attempt+1}/{max_retries}. Retrying in {wait_s}s...")
            time.sleep(wait_s)
        except ClientError:
            raise

    raise last_err if last_err is not None else ValueError("Gemini image generation failed with no usable response")


def realize_single_layer_fullframe(
    image,
    layer_context,
    ordered_contexts,
    depth_fallback_masks,
    layer_guidance,
    output_dir,
    model_name,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    order = layer_context["order"]
    layer_name = layer_context["layer_name"]

    mask_pack = build_control_masks_for_layer(layer_context, ordered_contexts, depth_fallback_masks)

    focused_input = apply_focus_mask_fullframe(
        image_rgb=image,
        show_mask=mask_pack["show_mask"],
        black_mask=mask_pack["black_mask"],
        beige_value=235,
    )

    prompt = build_fullframe_prompt(layer_context, layer_guidance)

    generated_full = gemini_edit(
        focused_input,
        prompt=prompt,
        model_name=model_name,
    )

    target_h, target_w = focused_input.shape[:2]
    if generated_full.shape[:2] != (target_h, target_w):
        generated_full = np.array(
            Image.fromarray(generated_full.astype(np.uint8)).resize(
                (target_w, target_h),
                Image.Resampling.LANCZOS,
            )
        )

    if HARD_ENFORCE_BLACK_AFTER_GENERATION:
        generated_full[mask_pack["black_mask"] > 0] = 0

    export_mask = mask_pack["current_support"] > 0

    generated_full_path = output_dir / f"{order:02d}_{layer_name}_generated_full.png"
    generated_visible_full_path = output_dir / f"{order:02d}_{layer_name}_generated_visible_full.png"
    focused_input_path = output_dir / f"{order:02d}_{layer_name}_focused_input.png"
    export_mask_path = output_dir / f"{order:02d}_{layer_name}_export_mask.png"
    prompt_path = output_dir / f"{order:02d}_{layer_name}_prompt.txt"

    Image.fromarray(focused_input).save(focused_input_path)
    Image.fromarray(generated_full).save(generated_full_path)
    Image.fromarray(mask_to_rgba_fullframe(generated_full, export_mask)).save(generated_visible_full_path)
    save_bool_mask(export_mask, export_mask_path)
    prompt_path.write_text(prompt, encoding="utf-8")

    return {
        "layer_name": layer_name,
        "order": order,
        "focused_input_path": str(focused_input_path),
        "generated_full_path": str(generated_full_path),
        "generated_visible_full_path": str(generated_visible_full_path),
        "export_mask_path": str(export_mask_path),
        "prompt_path": str(prompt_path),
    }

In [ ]:
# =========================
# FULL-FRAME GENERATION
# =========================

if IMAGE_GENERATION_PROVIDER != "gemini":
    raise NotImplementedError("This notebook currently realizes the final full-frame layers with Gemini image generation.")

fullframe_results = []
failed_layers = []

for ctx in ordered_contexts:
    print("\n==============================")
    print("Processing full-frame layer:", ctx["layer_name"])
    print("==============================")

    try:
        result = realize_single_layer_fullframe(
            image=image,
            layer_context=ctx,
            ordered_contexts=ordered_contexts,
            depth_fallback_masks=depth_fallback_masks,
            layer_guidance=layer_guidance_map[ctx["layer_name"]],
            output_dir=FULLFRAME_OUTPUT_DIR,
            model_name=GEMINI_IMAGE_MODEL,
        )
        fullframe_results.append(result)
        print("Saved:")
        print("  focused_input      :", result["focused_input_path"])
        print("  generated_full     :", result["generated_full_path"])
        print("  generated_visible  :", result["generated_visible_full_path"])
        print("  prompt             :", result["prompt_path"])

    except Exception as e:
        failed_layers.append({"layer_name": ctx["layer_name"], "error": repr(e)})
        print(f"FAILED layer {ctx['layer_name']}: {e}")

print("\nCompleted layers:", len(fullframe_results))
print("Failed layers:", len(failed_layers))
if failed_layers:
    print(json.dumps(failed_layers, indent=2))


# =========================
# VISUALIZE RESULTS
# =========================

for result in fullframe_results:
    print(result["layer_name"])
    for name, p in [
        ("focused_input", result["focused_input_path"]),
        ("generated_full", result["generated_full_path"]),
        ("generated_visible_full", result["generated_visible_full_path"]),
    ]:
        img = Image.open(p)
        plt.figure(figsize=(8, 10))
        plt.imshow(img)
        plt.title(f'{result["layer_name"]} — {name}')
        plt.axis("off")
        plt.show()


# =========================
# ZIP OUTPUTS
# =========================

output_zip_base = Path("/content") / f"{SCENE_NAME}_depth_band_fullframe_outputs"
shutil.make_archive(str(output_zip_base), "zip", str(SCENE_DIR))

print(f"Compressed '{SCENE_DIR}' to '{output_zip_base}.zip'")
print("Download path:", f"{output_zip_base}.zip")